# ARE-004.1 - YummyAnime profile exploration
Run all cells with the local .venv kernel. Install `pip install -e ".[data,research,dev]"`.
Public X-Application only; at most four GETs for one profile, no retries or redirects.
Token comes from local .env/environment. No dataframe exports.

PRIVACY: display outputs contain personal data in memory; Jupyter may autosave them.
Clear ALL outputs before saving or sharing. Never commit an executed notebook.
HTTP, normalization and checks are implemented in src/anime_recommender/.


In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from anime_recommender.exploration import project_root, explore_profile
from anime_recommender.ingestion.yummyanime import YummyAnimeClient

root = project_root()
load_dotenv(root / ".env", override=False)
nickname = os.getenv("YUMMYANIME_TEST_NICKNAME", "").strip() or "First_Light"
client = YummyAnimeClient(os.environ.get("YUMMYANIME_APPLICATION_TOKEN", ""))

from anime_recommender.preprocessing.user_data import status_distribution
from anime_recommender.evaluation.user_data import (
    rating_summary, rating_distribution, user_quality_report, website_comparison, profile_summary,
)


## Fetch one profile and its lists
An API failure stops further requests and is displayed below.

In [ ]:
snapshot = explore_profile(client, nickname)
df = snapshot.frame
display(pd.DataFrame(client.request_log))
print("Complete:", snapshot.complete, "Error:", snapshot.error)
display(profile_summary(snapshot.profile, nickname))


## Lists, fields and missing values
Raw records remain unchanged. No deduplication, imputation or zero-rating conversion.

In [ ]:
quality = user_quality_report(snapshot.rows, df)
display(df.head(10))
display(quality["columns"])
display(quality["summary"])
display(quality["coercions"])


## Watch categories and favorites
IDs 0/1/2/3/5 mean Watching/Planned/Completed/Dropped/On hold.
Favorites filter 4 is not a watch status; null remains missing.
Malformed numeric fields become NA and are counted in coercions.


In [ ]:
statuses = status_distribution(df)
display(statuses)
display(quality["missing_status_shapes"])
display(snapshot.favorites)


## User ratings and MAL IDs
Only user.rating is analyzed; global anime rating is never substituted.
Positive values provisionally mean rated. Zero meaning unrated is not independently
confirmed. Matching profile statistics supports ownership, not per-item proof.


In [ ]:
ratings = rating_summary(df)
display(ratings)
display(rating_distribution(df))
display(snapshot.ratings)


## Manual website comparison
No scraping. Fill None counters manually, then restore None before saving.
Blank website counts never mean a match. Favorites overlap watch categories.


In [ ]:
website_counts = {"Completed": None, "Planned": None, "Watching": None,
                  "Dropped": None, "On hold": None, "Favorites": None}
comparison = website_comparison(df, website_counts, complete=snapshot.complete)
display(comparison)


## Limits
This is a snapshot, not status-change history. Check UI counters before asserting
completeness. No ALS or train/validation/test split. See notebook 02 for local
training-data statistics and MAL mapping. Clear outputs before saving.
